
# Quantum State Tomography using Machine Learning

This notebook demonstrates how to reconstruct a quantum state density matrix using:

- Neural Networks
- Bayesian Inference
- Compressed Sensing

The project simulates quantum measurements in the **X, Y, Z bases** and reconstructs the density matrix from measurement statistics.


In [ ]:

!pip install numpy matplotlib pandas scikit-learn scipy


## Import libraries

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from scipy.linalg import sqrtm


## Define Pauli matrices and utility functions

In [ ]:

I2 = np.array([[1,0],[0,1]],dtype=complex)
X = np.array([[0,1],[1,0]],dtype=complex)
Y = np.array([[0,-1j],[1j,0]],dtype=complex)
Z = np.array([[1,0],[0,-1]],dtype=complex)

PAULIS = {"X":X,"Y":Y,"Z":Z}

def bloch_to_density(rx,ry,rz):
    return 0.5*(I2 + rx*X + ry*Y + rz*Z)

def density_to_bloch(rho):
    rx = np.real(np.trace(rho@X))
    ry = np.real(np.trace(rho@Y))
    rz = np.real(np.trace(rho@Z))
    return np.array([rx,ry,rz])

def random_pure_state():
    u = np.random.rand()
    v = np.random.rand()
    theta = np.arccos(1-2*u)
    phi = 2*np.pi*v
    psi = np.array([np.cos(theta/2),np.exp(1j*phi)*np.sin(theta/2)])
    return psi

def statevector_to_density(psi):
    return np.outer(psi,np.conj(psi))


## Measurement simulation

In [ ]:

def projective_measurement_probs(rho,basis):
    sigma = PAULIS[basis]
    exp_val = np.real(np.trace(rho@sigma))
    p_plus = (1+exp_val)/2
    p_minus = 1-p_plus
    return p_plus,p_minus

def simulate_measurements(rho,shots=1000):
    results = {}
    features = []
    for basis in ["X","Y","Z"]:
        p_plus,p_minus = projective_measurement_probs(rho,basis)
        counts = np.random.multinomial(shots,[p_plus,p_minus])
        n_plus,n_minus = counts
        exp_est = (n_plus-n_minus)/shots

        results[basis] = {"n_plus":n_plus,"n_minus":n_minus,"exp_est":exp_est}

        features.extend([n_plus/shots,n_minus/shots,exp_est])
    return results,np.array(features)


## Generate synthetic training dataset

In [ ]:

X_data = []
y_data = []

for i in range(2000):
    psi = random_pure_state()
    rho = statevector_to_density(psi)

    _,features = simulate_measurements(rho,shots=500)
    bloch = density_to_bloch(rho)

    X_data.append(features)
    y_data.append(bloch)

X_data = np.array(X_data)
y_data = np.array(y_data)


## Train neural network tomography model

In [ ]:

X_train,X_test,y_train,y_test = train_test_split(X_data,y_data,test_size=0.2)

model = MLPRegressor(hidden_layer_sizes=(128,128),max_iter=500)
model.fit(X_train,y_train)

y_pred = model.predict(X_test)

mse = mean_squared_error(y_test,y_pred)
print("Test MSE:",mse)


## Reconstruct quantum state from measurements

In [ ]:

psi_true = random_pure_state()
rho_true = statevector_to_density(psi_true)

measurement_results,features_test = simulate_measurements(rho_true,shots=1000)

bloch_pred = model.predict(features_test.reshape(1,-1))[0]

norm = np.linalg.norm(bloch_pred)
if norm>1:
    bloch_pred = bloch_pred/norm

rho_pred = bloch_to_density(*bloch_pred)

print("True Bloch:",density_to_bloch(rho_true))
print("Predicted Bloch:",bloch_pred)


## Compute fidelity

In [ ]:

def fidelity(rho,sigma):
    sr = sqrtm(rho)
    middle = sr@sigma@sr
    sm = sqrtm(middle)
    return np.real(np.trace(sm))**2

print("Fidelity:",fidelity(rho_true,rho_pred))


## Visualize density matrices

In [ ]:

fig,axes = plt.subplots(1,2,figsize=(8,4))

axes[0].imshow(np.real(rho_true),cmap='coolwarm')
axes[0].set_title("True Density Matrix")

axes[1].imshow(np.real(rho_pred),cmap='coolwarm')
axes[1].set_title("Predicted Density Matrix")

plt.show()



## Observations

- The neural network learns the mapping from measurement statistics to Bloch vectors.
- Fidelity values close to **1** indicate accurate reconstruction.
- With higher shot counts and larger datasets, the reconstruction accuracy improves.

This framework can be extended to:

- **2‑qubit tomography**
- **entangled state reconstruction**
- **quantum noise analysis**
- **experimental quantum hardware data**
